<a href="https://colab.research.google.com/github/shin584/project/blob/model_analysis/SpCas9_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%%writefile /content/nt_v3.py
import os; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # 모델 다운로드 표시줄 끄기
import argparse, inspect, os, numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback, set_seed)

ap = argparse.ArgumentParser()
ap.add_argument("--seed", type=int, default=42); ap.add_argument("--epochs", type=int, default=5)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--data", default=os.environ.get("DATA", "/content")); ap.add_argument("--out", default=os.environ.get("OUT", "/content"))
ap.add_argument("--model", default="InstaDeepAI/nucleotide-transformer-500m-human-ref")
ap.add_argument("--sub", type=int, default=0)
a = ap.parse_args(); set_seed(a.seed); os.makedirs(a.out, exist_ok=True)

H = ['SpCas9','SpCas9-NG','VRQR variant','xCas','Sniper-Cas9','SpCas9-HF.1','eSpCas9(1.1)','HypaCas9','evoCas9']
tr = pd.read_csv(f"{a.data}/train.csv"); va = pd.read_csv(f"{a.data}/val.csv"); te = pd.read_csv(f"{a.data}/test.csv")
if a.sub: tr, va, te = tr.sample(a.sub, random_state=0), va.sample(a.sub//5, random_state=0), te.sample(a.sub//5, random_state=0)
for d in (tr, va, te): assert d.sequence.str.len().eq(30).all()
print("train", len(tr), "val", len(va), "test", len(te))

Ytr = tr[H].values.astype(np.float32) / 100.0
mu = np.nanmean(Ytr, 0); sd = np.nanstd(Ytr, 0)                 # 헤드별 표준화 (측정된 값만)
cov = (~np.isnan(Ytr)).mean(0).astype(np.float32)               # 헤드별 측정 비율 (손실 균형용)

tok = AutoTokenizer.from_pretrained(a.model)
class DS(torch.utils.data.Dataset):
    def __init__(s, d):
        s.enc = tok(list(d.sequence), padding="max_length", truncation=True, max_length=32)
        y = d[H].values.astype(np.float32) / 100.0
        m = ~np.isnan(y)
        s.lab = np.hstack([np.nan_to_num((y - mu) / sd), m.astype(np.float32)]).astype(np.float32)   # [표준화 label 9, mask 9]
    def __len__(s): return len(s.lab)
    def __getitem__(s, i):
        x = {k: torch.tensor(v[i]) for k, v in s.enc.items()}
        x["labels"] = torch.tensor(s.lab[i]); return x

class MaskedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        lab = inputs.pop("labels"); out = model(**inputs); p = out.logits
        y, m = lab[:, :9], lab[:, 9:]
        c = torch.tensor(cov, device=p.device)
        loss = (m * (y - p) ** 2 / c).sum(-1).mean()               # 결측 헤드는 손실 0
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(a.model, num_labels=9, problem_type="regression")
k = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
args = TrainingArguments(output_dir=f"/content/out_ntv3_{a.seed}", num_train_epochs=a.epochs,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, learning_rate=a.lr, weight_decay=0.01,
    fp16=torch.cuda.is_available(), save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False, logging_steps=200, report_to="none",
    disable_tqdm=True, seed=a.seed, **{k: "epoch"})
dtr, dva, dte = DS(tr), DS(va), DS(te)
t = MaskedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
t.train()

def predict(ds): return t.predict(ds).predictions * sd + mu    # 원래 척도(/100)로 복원, (N,9)
def evaluate(P, d):
    y = d[H].values / 100.0; ym = ~np.isnan(y)
    gg = (d.sequence.str[25:27].values == "GG"); r = {}; a_, g_ = [], []
    for j in range(9):
        ok = ym[:, j]
        a_.append(spearmanr(P[ok, j], y[ok, j])[0])
        g = ok & gg; g_.append(spearmanr(P[g, j], y[g, j])[0] if g.sum() >= 50 else np.nan)
        if j == 0: r["SpCas9_MAE"] = np.abs(P[ok, 0] - y[ok, 0]).mean()
    r.update(Sp_all=a_[0], Sp_ngg=g_[0], mean_all=np.nanmean(a_), mean_ngg=np.nanmean(g_))
    return r
Pva, Pte = predict(dva), predict(dte)
rv, rt = evaluate(Pva, va), evaluate(Pte, te)
row = dict(cfg="NT_v3", seed=a.seed, val_Sp_ngg=rv["Sp_ngg"], **{"test_" + k_: v for k_, v in rt.items()})
print({k_: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k_, v in row.items()})
pd.DataFrame([row]).to_csv(f"{a.out}/nt_v3_seed{a.seed}_results.csv", index=False)

M = f"{a.out}/nt_v3_seed{a.seed}"                               # 모델 + 토크나이저 + 복원용 mu/sd 저장
t.save_model(M); tok.save_pretrained(M); np.save(f"{M}/mu_sd.npy", np.stack([mu, sd]))
print("저장 완료:", M)

Overwriting /content/nt_v3.py


In [6]:
!python /content/nt_v3.py --seed 42 --data "/content/drive/MyDrive/SpCas9" --out "/content/drive/MyDrive/SpCas9/spcas9_v3"

train 29134 val 3676 test 5186
[transformers] EsmForSequenceClassification LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '5.605', 'grad_norm': '37.23', 'learning_rate': '1.913e-05', 'epoch': '0.2195'}
{'lo

In [8]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

M = "/content/drive/MyDrive/SpCas9/spcas9_v3/nt_v3_seed42"
H = [
    "SpCas9",
    "SpCas9-NG",
    "VRQR variant",
    "xCas",
    "Sniper-Cas9",
    "SpCas9-HF.1",
    "eSpCas9(1.1)",
    "HypaCas9",
    "evoCas9",
]

tok = AutoTokenizer.from_pretrained(M)
model = AutoModelForSequenceClassification.from_pretrained(M).eval().cuda()
mu, sd = np.load(f"{M}/mu_sd.npy")  # 헤드별 평균/표준편차 (9개씩)


def predict(seqs):
    """30bp 서열 리스트 -> (N, 9) 효율 (0~1 척도, 즉 %/100)"""
    out = []
    for i in range(0, len(seqs), 64):
        enc = tok(
            seqs[i : i + 64],
            padding="max_length",
            truncation=True,
            max_length=32,
            return_tensors="pt",
        ).to("cuda")
        with torch.no_grad():
            out.append(model(**enc).logits.float().cpu().numpy())
    return np.concatenate(out) * sd + mu  # 표준화 되돌리기


p = predict(["ACGT" + "ACGT" * 5 + "AGG" + "TCA"])
print(dict(zip(H, p[0].round(3))))

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{'SpCas9': np.float32(0.502), 'SpCas9-NG': np.float32(0.369), 'VRQR variant': np.float32(0.336), 'xCas': np.float32(0.367), 'Sniper-Cas9': np.float32(0.489), 'SpCas9-HF.1': np.float32(0.34), 'eSpCas9(1.1)': np.float32(0.408), 'HypaCas9': np.float32(0.434), 'evoCas9': np.float32(0.104)}


## ISM 전수 산출 (#132 · 티켓 C1)

3,851행 x 91 = 약 350K forward. 계산 로직은 리포의 `v3_ism.py`(로컬 테스트됨)에 있고 이 노트북은 드라이버다.

**먼저 할 일 — 모듈 3개를 Drive에 복사한다.** `MODULES` 폴더(`MyDrive/SpCas9/spcas9_v3/code`)에 넣는다:

- `v3_ism.py`, `v3_predict.py`, `v3_strata.py` — `CODE_REF` 커밋의 것
- `split_checksum.json` — `xai_test/SpCas9/published_split/`에서

clone도 토큰도 쓰지 않는다. 대신 셀이 세 파일의 sha256을 `CODE_REF` 커밋의 값과 대조하므로, 구버전을 올려 두면 **GPU 시간을 쓰기 전에** 멈춘다. 줄끝(CRLF/LF)은 정규화해 비교하니 Windows 작업 트리에서 그대로 복사해도 된다. 모듈이 바뀌면 `CODE_REF`와 `MODULE_SHA256`을 함께 갱신한다.

그 밖에:

- `transformers` 설치 뒤 런타임 재시작을 요구할 수 있다. 재시작한 뒤 드라이브 마운트 셀과 이 셀만 다시 실행하면 된다.
- **OOM이면 `ISM_BATCH_SIZE`만 낮추고 같은 셀을 다시 실행한다.** 끝난 샤드는 `SHARD_DIR`에 남아 있어 다시 돌지 않는다 — 세션이 끊긴 경우도 같다.
- 산출물은 `artifacts/v3_ism/ism.npz` + `ism.sidecar.json`. head 9개가 전부 저장되지만 분석은 3개이고, 사이드카가 나머지 6개를 "저장했으나 분석하지 않음"으로 적는다. 사이드카의 `code_ref_verified_by`는 핀을 체크아웃이 아니라 파일 digest로 확인했음을 남긴다.


In [ ]:
# ISM 전수 산출 드라이버 (#132). 계산은 리포의 v3_ism.py, 이 셀은 드라이버다.
CODE_REF = "8a02f86c8dfba8fc218b8820124b2e6963f7dc96"  # v3_ism.py가 들어간 커밋 (브랜치 이름은 거부된다)
TRANSFORMERS_VERSION = "5.8.1"
ISM_BATCH_SIZE = 256  # OOM이면 이 값만 낮춘다. 사이드카에 기록된다.
ISM_SHARD_ROWS = 256  # 샤드 하나가 끝날 때마다 Drive에 저장된다.

DRIVE = "/content/drive/MyDrive/SpCas9"
CHECKPOINT = f"{DRIVE}/spcas9_v3/nt_v3_seed42"
SPLIT_CSV = f"{DRIVE}/test.csv"
SHARD_DIR = f"{DRIVE}/spcas9_v3/v3_ism_shards"  # 끊기면 여기서 이어간다
OUT_DIR = f"{DRIVE}/spcas9_v3/artifacts/v3_ism"
MODULES = f"{DRIVE}/spcas9_v3/code"  # 여기에 모듈 3개를 복사해 둔다

import hashlib
import json
import re
import sys
from pathlib import Path

assert re.fullmatch(r"[0-9a-f]{40}", CODE_REF), "CODE_REF를 커밋 sha 40자로 채운다"

!pip install -q "transformers=={TRANSFORMERS_VERSION}"

# 모듈 3개를 Drive에서 import한다. clone도 토큰도 쓰지 않는다 — MODULES 폴더에
# CODE_REF 커밋의 v3_ism.py / v3_predict.py / v3_strata.py 를 복사해 둔다.
#
# 체크아웃으로 핀을 확인할 수 없으니 **파일 내용**으로 확인한다. 아래 값은 그
# 커밋이 담고 있는 세 파일의 sha256이고, 줄끝(CRLF/LF) 차이는 정규화해 비교하므로
# Windows 작업 트리에서 그대로 복사해도 된다. 구버전을 올려 두면 여기서 멈춘다 —
# 한 시간짜리 런을 돌린 뒤에 알아차리는 것보다 싸다.
MODULE_SHA256 = {
    "v3_ism.py": "beef1e1327bc60faeb57cf2589bcfb952752b08b537f66d6154087c1f56a0c56",
    "v3_predict.py": "fb18ba0c570e965d75d1f7f9532fc42c8fe1e955d086262421409470530f38fc",
    "v3_strata.py": "b381e57162520e2af5ecad42e47d8eceedc5a6b550944b40e7ccfb6557a1a9e3",
}


def module_digest(path):
    """파일 내용의 sha256 = 그 커밋 blob의 sha256.

    텍스트 모드로 읽으므로 줄끝은 LF로 정규화된다 (universal newlines):
    Windows 작업 트리의 CRLF 사본도 같은 값이 나온다.
    """
    text = Path(path).read_text(encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


missing = [n for n in MODULE_SHA256 if not Path(f"{MODULES}/{n}").exists()]
assert not missing, f"{MODULES} 에 {missing} 가 없다"
MODULE_DIGESTS = {name: module_digest(f"{MODULES}/{name}") for name in MODULE_SHA256}
stale = [n for n, d in MODULE_DIGESTS.items() if d != MODULE_SHA256[n]]
assert not stale, (
    f"{stale} 의 내용이 CODE_REF {CODE_REF[:12]} 와 다르다: "
    f"그 커밋의 파일을 {MODULES} 에 다시 복사한다"
)

sys.path.insert(0, MODULES)
import numpy as np
import pandas as pd
import transformers
import v3_ism
import v3_predict
import v3_strata
from transformers import AutoModelForSequenceClassification, AutoTokenizer

assert transformers.__version__ == TRANSFORMERS_VERSION, (
    f"transformers {transformers.__version__} != {TRANSFORMERS_VERSION}: "
    "런타임을 재시작하고 이 셀을 다시 실행한다"
)

# 소비하는 split이 리뷰된 그 split인지. 체크섬은 리포에 고정돼 있다.
with open(f"{MODULES}/split_checksum.json", encoding="utf-8") as handle:
    expected = json.load(handle)["files"]["test.csv"]
assert v3_predict.sha256_of(SPLIT_CSV) == expected, (
    "Drive의 test.csv가 리포에 고정된 split과 다르다"
)

frame = v3_strata.analysis_intersection(pd.read_csv(SPLIT_CSV))
assert len(frame) == 3851, f"분석 교집합이 {len(frame)}행 (3,851 기대)"

mu, sd = v3_predict.load_mu_sd(CHECKPOINT)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT).eval().cuda()
# forward_fn의 배치와 run_ism의 배치는 같아야 한다: 사이드카에 적히는 값이
# GPU가 실제로 본 배치여야 OOM 보고를 그대로 믿을 수 있다.
forward_fn = v3_predict.model_forward_fn(model, tokenizer, batch_size=ISM_BATCH_SIZE)

result = v3_ism.run_ism(
    forward_fn,
    frame["sequence"].tolist(),
    frame["row_index"].to_numpy(),
    mu=mu,
    sd=sd,
    batch_size=ISM_BATCH_SIZE,
    shard_rows=ISM_SHARD_ROWS,
    checkpoint_dir=SHARD_DIR,
    progress=lambda event: print(
        f"shard {event['shard']}: {event['rows_done']}/{event['rows_total']} rows"
        + (" (reused)" if event["reused"] else ""),
        flush=True,
    ),
)

path = v3_ism.write_ism_artifact(
    OUT_DIR,
    result,
    code_ref=CODE_REF,
    split_csv=SPLIT_CSV,
    checkpoint_dir=CHECKPOINT,
    # 핀을 어떻게 확인했는지 사이드카에 남긴다: 체크아웃이 아니라 파일 digest다.
    extra={
        "code_ref_verified_by": "module_sha256",
        "module_sha256": MODULE_DIGESTS,
    },
)
sidecar = json.loads((path.parent / "ism.sidecar.json").read_text(encoding="utf-8"))
print(path, result.delta.shape, result.delta.dtype)
print("reused shards:", len(result.reused_shards), "| forwards:", sidecar["forwards"])
print("finite:", bool(np.isfinite(result.delta).all()))
print("delta sha256:", sidecar["delta_sha256"])

## IG 400행 산출 (#133 · 티켓 D)

400행 x 200스텝 x 3 head ~ 240K forward+backward. 계산 로직은 리포의 `v3_ig.py`(로컬 테스트됨)에 있고 이 노트북은 드라이버다. 명단은 **#131이 고정한 400행을 디스크에서 읽는다** — 노트북에서 다시 뽑지 않는다(AC7).

**먼저 할 일 — 파일 7개를 Drive에 복사한다.**

`MODULES` 폴더(`MyDrive/SpCas9/spcas9_v3/code`)에:

- `v3_ig.py`, `v3_predict.py`, `v3_selection.py`, `v3_strata.py` — `CODE_REF` 커밋의 것
- `split_checksum.json` — `xai_test/SpCas9/published_split/`에서

`SUBSET_DIR` 폴더(`MyDrive/SpCas9/spcas9_v3/artifacts/v3_ig_subset`)에:

- `ig_subset.json`, `ig_subset_checksum.json` — `artifacts/v3_ig_subset/`에서 **쌍으로**

ISM 드라이버와 같이 clone도 토큰도 쓰지 않는다. 셀이 모듈 5개의 sha256을 `CODE_REF` 커밋의 값과 대조하고, 명단은 **체크섬 파일에 적힌 값까지** 리포에 핀으로 박힌 값과 맞춰 보므로 — 구버전 쌍을 통째로 올려도 `load_ig_subset`의 바이트 대조는 통과하기 때문이다 — **GPU 시간을 쓰기 전에** 멈춘다. 줄끝(CRLF/LF)은 정규화해 비교하니 Windows 작업 트리에서 그대로 복사해도 된다.

그 밖에:

- **OOM이면 `IG_BATCH_SIZE`만 낮추고 같은 셀을 다시 실행한다.** IG는 스텝마다 autograd 그래프를 들고 head당 backward를 한 번씩 돌기 때문에 forward 전용 배치보다 훨씬 작다(기본 8).
- ISM과 달리 **샤드가 없다**: 한 번의 호출로 끝내고 중간 상태를 디스크에 남기지 않는다. 끊기면 240K짜리 런을 처음부터 다시 돌리는데, ISM의 350K와 달리 그 값이 재개 로직의 복잡도보다 싸다. 대신 배치마다 진행률을 찍으므로 멈춘 것과 느린 것을 구분할 수 있다.
- 산출물은 `artifacts/v3_ig/ig.npz` + `ig.sidecar.json` **하나**다. 400행 전체가 한 아카이브에 `row_index` 오름차순으로 들어가고, 층은 사이드카의 `strata`와 명단이 말한다 — 층당 200행씩 둘로 쪼개면 사이드카 두 개가 서로 동의할 수밖에 없는 내용을 적고, #134의 join이 아무것도 고정하지 않은 순서로 둘을 이어 붙이게 된다.
- **스텝은 200이다.** 50스텝(SaCas9 파이프라인 값)으로 돌린 첫 런이 통과율 0.85, 최악 잔차 0.69(표준화 단위, sd를 곱하면 ~15%p)였고, 같은 체크포인트로 최악 3행을 200스텝에서 다시 재면 0.0026/0.0194/0.0026까지 떨어졌다 — 적분오차였다. 비용은 4배(240K)다.
- **완결성은 보고만 하고 런을 중단하지 않는다**(ADR 0006). 통과율과 `f_baseline`이 사이드카에 남아 Colab 세션이 끝난 뒤에도 검사를 재현할 수 있다. 반면 **전부-0 attribution은 쓰기가 거부한다** — 그건 SaCas9 Model B에서 실제 체크포인트로 나온 배선 결함이고, 신호가 약한 행과 다르다.


In [ ]:
# IG 400행 산출 드라이버 (#133). 계산은 리포의 v3_ig.py, 이 셀은 드라이버다.
CODE_REF = "a1999e36b693caf4864838baeb9fc03ebd08f2a8"  # 위 모듈들이 들어간 커밋 (브랜치 이름은 거부된다)
TRANSFORMERS_VERSION = "5.8.1"
IG_STEPS = 200  # 50스텝은 완결성 통과율 0.85였다 (v3_ig.DEFAULT_IG_STEPS 주석 참조)
IG_BATCH_SIZE = 8  # OOM이면 이 값만 낮춘다. 사이드카에 기록된다.

DRIVE = "/content/drive/MyDrive/SpCas9"
CHECKPOINT = f"{DRIVE}/spcas9_v3/nt_v3_seed42"
SPLIT_CSV = f"{DRIVE}/test.csv"
SUBSET_DIR = f"{DRIVE}/spcas9_v3/artifacts/v3_ig_subset"  # 명단을 여기서 읽는다
OUT_DIR = f"{DRIVE}/spcas9_v3/artifacts/v3_ig"
MODULES = f"{DRIVE}/spcas9_v3/code"  # 여기에 모듈 5개를 복사해 둔다

import json
import re
import sys
from hashlib import sha256  # 이름으로 가져온다: 이 셀만 다시 실행해도 선다
from pathlib import Path

assert re.fullmatch(r"[0-9a-f]{40}", CODE_REF), "CODE_REF를 커밋 sha 40자로 채운다"

!pip install -q "transformers=={TRANSFORMERS_VERSION}"

# 모듈 5개를 Drive에서 import한다. clone도 토큰도 쓰지 않는다 — MODULES 폴더에
# CODE_REF 커밋의 파일들을 복사해 둔다. 체크아웃으로 핀을 확인할 수 없으니
# **파일 내용**으로 확인한다. 줄끝(CRLF/LF) 차이는 정규화해 비교하므로 Windows
# 작업 트리에서 그대로 복사해도 된다. 구버전을 올려 두면 여기서 멈춘다.
MODULE_SHA256 = {
    "v3_ig.py": "54acfcccc1606ee23345c5c65896e8bcbd3cd6c26b1e1c850edc21a42afa83d5",
    "v3_predict.py": "fb18ba0c570e965d75d1f7f9532fc42c8fe1e955d086262421409470530f38fc",
    "v3_selection.py": "47fd72d5845550a1900477854533b1a03de995c86007d2bdf78e1736aba18f1d",
    "v3_strata.py": "b381e57162520e2af5ecad42e47d8eceedc5a6b550944b40e7ccfb6557a1a9e3",
}

# 명단의 핀. `load_ig_subset`은 ig_subset.json 을 **옆에 있는 체크섬 파일**과
# 대조하므로, 구버전 쌍을 통째로 올리면 통과한다. 그래서 체크섬 파일이 적고 있는
# 값 자체를 리포에 고정된 값과 먼저 맞춰 본다 — 400행이 #131이 뽑은 그 400행인지는
# 240K backward를 쓰기 전에 끝나야 하는 질문이다.
IG_SUBSET_SHA256 = "8193073ea8dbd068344e1b3d48117a4b94c8431e5bbc3c5c7f4a10070fc785e6"


def module_digest(path):
    """파일 내용의 sha256 = 그 커밋 blob의 sha256.

    텍스트 모드로 읽으므로 줄끝은 LF로 정규화된다 (universal newlines):
    Windows 작업 트리의 CRLF 사본도 같은 값이 나온다.
    """
    text = Path(path).read_text(encoding="utf-8")
    return sha256(text.encode("utf-8")).hexdigest()


missing = [n for n in MODULE_SHA256 if not Path(f"{MODULES}/{n}").exists()]
assert not missing, f"{MODULES} 에 {missing} 가 없다"
MODULE_DIGESTS = {name: module_digest(f"{MODULES}/{name}") for name in MODULE_SHA256}
stale = [n for n, d in MODULE_DIGESTS.items() if d != MODULE_SHA256[n]]
assert not stale, (
    f"{stale} 의 내용이 CODE_REF {CODE_REF[:12]} 와 다르다: "
    f"그 커밋의 파일을 {MODULES} 에 다시 복사한다"
)

sys.path.insert(0, MODULES)
import numpy as np
import pandas as pd
import transformers
import v3_ig
import v3_predict
import v3_selection
import v3_strata
from transformers import AutoModelForSequenceClassification, AutoTokenizer

assert transformers.__version__ == TRANSFORMERS_VERSION, (
    f"transformers {transformers.__version__} != {TRANSFORMERS_VERSION}: "
    "런타임을 재시작하고 이 셀을 다시 실행한다"
)

# 소비하는 split이 리뷰된 그 split인지. 체크섬은 리포에 고정돼 있다.
with open(f"{MODULES}/split_checksum.json", encoding="utf-8") as handle:
    expected_split = json.load(handle)["files"]["test.csv"]
assert v3_predict.sha256_of(SPLIT_CSV) == expected_split, (
    "Drive의 test.csv가 리포에 고정된 split과 다르다"
)

# 명단: 체크섬 파일이 적은 값 -> 리포의 핀, 그 다음 load_ig_subset이 바이트 대조.
subset_path = f"{SUBSET_DIR}/{v3_selection.IG_SUBSET_NAME}"
checksum_path = Path(f"{SUBSET_DIR}/{v3_selection.IG_SUBSET_CHECKSUM_NAME}")
assert checksum_path.exists(), f"{checksum_path} 가 없다 — 명단은 쌍으로 복사한다"
recorded = json.loads(checksum_path.read_text(encoding="utf-8"))["files"]
assert recorded.get(v3_selection.IG_SUBSET_NAME) == IG_SUBSET_SHA256, (
    "Drive의 ig_subset.json이 리포에 고정된 명단이 아니다: "
    f"체크섬 파일은 {recorded.get(v3_selection.IG_SUBSET_NAME)} 를 적고 있는데 "
    f"리포의 핀은 {IG_SUBSET_SHA256} 다"
)
subset = v3_selection.load_ig_subset(subset_path)
rows = np.sort(subset.row_index)
assert rows.size == v3_selection.IG_SUBSET_ROWS, f"명단이 {rows.size}행 (400 기대)"

# 서열은 split에서 가져온다. 명단에 있는 row_index가 분석 교집합에 없으면
# .loc 이 KeyError를 던진다 — 조용히 399행으로 줄어드는 쪽보다 낫다.
frame = v3_strata.analysis_intersection(pd.read_csv(SPLIT_CSV))
chosen = frame.set_index("row_index").loc[rows]
sequences = chosen["sequence"].tolist()
print(f"{len(sequences)}행 / 명단 {rows.size}행 / 교집합 {len(frame)}행")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT).eval().cuda()

result = v3_ig.run_ig(
    model,
    tokenizer,
    sequences,
    rows,
    heads=v3_strata.ANALYSED_HEADS,  # backward가 head당 필요해 예산이 3개로 고정됐다
    steps=IG_STEPS,
    batch_size=IG_BATCH_SIZE,
    progress=lambda event: print(
        f"{event['rows_done']}/{event['rows_total']} rows", flush=True
    ),
)

# 완결성은 **보고**다 (ADR 0006): 통과율을 찍고 런을 중단하지 않는다. 사이드카가
# 같은 보고를 담으므로 세션이 끝난 뒤에도 같은 수치를 읽을 수 있다.
report = v3_ig.completeness_report(
    result.completeness_residual, result.f_x, result.f_baseline
)
print(
    f"completeness {report['passing']}/{report['checks']} "
    f"(pass rate {report['pass_rate']:.4f}, max |residual| {report['max_abs_residual']:.4g})"
)
print("per-head pass rate:", dict(zip(result.heads, report["per_head"]["pass_rate"])))

path = v3_ig.write_ig_artifact(
    OUT_DIR,
    result,
    code_ref=CODE_REF,
    split_csv=SPLIT_CSV,
    subset_manifest=subset_path,  # 경로로 넘긴다: 체크섬을 다시 확인한다
    checkpoint_dir=CHECKPOINT,
    # 핀을 어떻게 확인했는지 사이드카에 남긴다: 체크아웃이 아니라 파일 digest다.
    extra={
        "code_ref_verified_by": "module_sha256",
        "module_sha256": MODULE_DIGESTS,
        "batch_size": IG_BATCH_SIZE,
    },
)
sidecar = json.loads((path.parent / "ig.sidecar.json").read_text(encoding="utf-8"))
print(path, result.attribution.shape, result.attribution.dtype)
print("stratum:", sidecar["stratum"], "| heads:", sidecar["heads"])
print("finite:", bool(np.isfinite(result.attribution).all()))
print("nonzero per head:", np.abs(result.attribution).sum(axis=(0, 2)).tolist())
print("attribution sha256:", sidecar["attribution_sha256"])

## Attention Rollout 전수 산출 (#136 · 티켓 C2)

3,851 forward. 계산 로직은 리포의 `v3_rollout.py`(로컬 테스트됨)에 있고 이 노트북은 드라이버다. **선행 티켓 없음** — ISM(#132)이나 IG(#133)를 기다리지 않는다.

**이 산출물에는 head 축이 없다.** head 9개는 공유 encoder trunk에 얹혀 있고 rollout은 그 trunk의 어텐션만의 함수이므로, 한 행에 `(5, 5)` 하나다 — 9장이 아니다(ADR 0008). head별 PAM 사용 차이(분석 질문 2)는 ISM과 IG가 답한다. 사이드카의 `head_invariance_note`가 이 사실을 명시한다.

**염기 좌표 30 × 30으로 펼치지 않는다**(ADR 0007). 어텐션은 토큰 간 양이고 마지막 토큰이 PAM(24-26)과 우flank(27-29)를 함께 묶으므로, 펼치면 그 ADR이 분리한 것을 다시 섞는다. SaCas9는 36 × 36으로 펼쳤지만 승계하지 않으며, 애초에 호출할 투영 함수가 모듈에 없다.

**모듈 5개를 Drive에 복사한다.** `MODULES` 폴더(`MyDrive/SpCas9/spcas9_v3/code`)에 `CODE_REF` 커밋의 것으로 넣는다: `v3_rollout.py`, `v3_ig.py`, `v3_predict.py`, `v3_selection.py`, `v3_strata.py`. ISM 셀보다 두 개 많은 이유는 `v3_rollout`이 토큰 계약(ADR 0007)을 `v3_ig`에서 가져오기 때문이다 — 사본을 두 개 두지 않는다. `split_checksum.json`도 같은 폴더에 둔다. 셀이 다섯 파일의 sha256을 대조하므로 구버전이면 **GPU 시간을 쓰기 전에** 멈춘다.

그 밖에:

- **체크포인트를 `attn_implementation="eager"`로 로드한다.** sdpa 같은 융합 커널은 어텐션 확률 행렬을 아예 만들지 않고, transformers 5.x는 그런 모델에 `output_attentions=True`를 주면 **에러 없이 빈 튜플**을 돌려준다 — 굴릴 것이 없는 rollout이 된다. `run_rollout`이 실행 중 eager를 강제하긴 하지만, 드라이버가 처음부터 eager로 로드해 두면 셀 안에서 따로 들여다보는 경우까지 같은 경로를 본다.
- **샤드도 재개도 없다.** ISM(#132)은 ~350K forward라 샤드로 끊어 저장하지만 이 런은 3,851 forward로 몇 분이다. 끊기면 셀을 그냥 다시 실행한다.
- OOM이면 `ROLLOUT_BATCH_SIZE`만 낮춘다. 배치 하나의 어텐션 텐서(24 레이어 × 20 head × 32 × 32)가 피크 메모리를 정한다.
- 산출물은 `artifacts/v3_rollout/rollout.npz` + `rollout.sidecar.json`. 키는 `rollout` `(3851, 5, 5)` float32와 `row_index` int64 **둘뿐**이다 — 레이어별 24장은 저장하지 않는다.

In [ ]:
# Attention Rollout 전수 산출 드라이버 (#136). 계산은 리포의 v3_rollout.py, 이 셀은 드라이버다.
CODE_REF = ""  # v3_rollout.py가 들어간 커밋 sha 40자로 채운다 (브랜치 이름은 거부된다)
TRANSFORMERS_VERSION = "5.8.1"
ROLLOUT_BATCH_SIZE = 64  # OOM이면 이 값만 낮춘다. 사이드카에 기록된다.

DRIVE = "/content/drive/MyDrive/SpCas9"
CHECKPOINT = f"{DRIVE}/spcas9_v3/nt_v3_seed42"
SPLIT_CSV = f"{DRIVE}/test.csv"
OUT_DIR = f"{DRIVE}/spcas9_v3/artifacts/v3_rollout"
MODULES = f"{DRIVE}/spcas9_v3/code"  # 여기에 모듈 5개를 복사해 둔다

import json
import re
import sys

# 이름으로 가져온다: 이 셀만 다시 실행해도 선다. 아래 noqa는 ruff가 노트북을
# 한 모듈로 읽어 ISM/IG 셀의 같은 import와 겹치기 때문이다 — 드라이버 셀은
# 셋 중 하나만 돌리므로 각자 자기 import를 가진다.
from hashlib import sha256  # noqa: F811
from pathlib import Path

assert re.fullmatch(r"[0-9a-f]{40}", CODE_REF), "CODE_REF를 커밋 sha 40자로 채운다"

!pip install -q "transformers=={TRANSFORMERS_VERSION}"

# v3_rollout의 import closure 전체. ISM 셀보다 두 개 많은 것은 v3_rollout이 토큰
# 계약(ADR 0007)을 v3_ig에서 가져오기 때문이다 — 사본을 두 개 두지 않는다.
# 체크아웃으로 핀을 확인할 수 없으니 **파일 내용**으로 확인한다. 줄끝(CRLF/LF)은
# 정규화해 비교하므로 Windows 작업 트리에서 그대로 복사해도 된다.
MODULE_SHA256 = {
    "v3_rollout.py": "d3c967fa29ebcd81ef1b88ce38ec1481573ccda1f19e8434cd25d2d88de78a8b",
    "v3_ig.py": "54acfcccc1606ee23345c5c65896e8bcbd3cd6c26b1e1c850edc21a42afa83d5",
    "v3_predict.py": "29b326c634f333ae2437b5e9d5d92382c94add69400d9ecbbb05ccbc9fa6b08b",
    "v3_selection.py": "772523529d732cbfc8dd8f727a653099624a7a3b2fe8f72cea5b74e9b07bfa2c",
    "v3_strata.py": "b381e57162520e2af5ecad42e47d8eceedc5a6b550944b40e7ccfb6557a1a9e3",
}


def module_digest(path):
    """파일 내용의 sha256 = 그 커밋 blob의 sha256.

    텍스트 모드로 읽으므로 줄끝은 LF로 정규화된다 (universal newlines).
    """
    text = Path(path).read_text(encoding="utf-8")
    return sha256(text.encode("utf-8")).hexdigest()


missing = [n for n in MODULE_SHA256 if not Path(f"{MODULES}/{n}").exists()]
assert not missing, f"{MODULES} 에 {missing} 가 없다"
MODULE_DIGESTS = {name: module_digest(f"{MODULES}/{name}") for name in MODULE_SHA256}
stale = [n for n, d in MODULE_DIGESTS.items() if d != MODULE_SHA256[n]]
assert not stale, (
    f"{stale} 의 내용이 CODE_REF {CODE_REF[:12]} 와 다르다: "
    f"그 커밋의 파일을 {MODULES} 에 다시 복사한다"
)

sys.path.insert(0, MODULES)
import numpy as np
import pandas as pd
import transformers
import v3_predict
import v3_rollout
import v3_strata
from transformers import AutoModelForSequenceClassification, AutoTokenizer

assert transformers.__version__ == TRANSFORMERS_VERSION, (
    f"transformers {transformers.__version__} != {TRANSFORMERS_VERSION}: "
    "런타임을 재시작하고 이 셀을 다시 실행한다"
)

# 소비하는 split이 리뷰된 그 split인지. 체크섬은 리포에 고정돼 있다.
with open(f"{MODULES}/split_checksum.json", encoding="utf-8") as handle:
    expected = json.load(handle)["files"]["test.csv"]
assert v3_predict.sha256_of(SPLIT_CSV) == expected, (
    "Drive의 test.csv가 리포에 고정된 split과 다르다"
)

frame = v3_strata.analysis_intersection(pd.read_csv(SPLIT_CSV))
assert len(frame) == 3851, f"분석 교집합이 {len(frame)}행 (3,851 기대)"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
# eager로 로드한다: 융합 커널(sdpa)은 어텐션 확률 행렬을 만들지 않고,
# transformers 5.x는 그런 모델에 output_attentions=True를 주면 에러 없이 빈
# 튜플을 돌려준다 — 굴릴 것이 없는 rollout이 된다.
model = (
    AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, attn_implementation="eager"
    )
    .eval()
    .cuda()
)

result = v3_rollout.run_rollout(
    model,
    tokenizer,
    frame["sequence"].tolist(),
    frame["row_index"].to_numpy(),
    batch_size=ROLLOUT_BATCH_SIZE,
    progress=lambda event: print(
        f"{event['rows_done']}/{event['rows_total']} rows", flush=True
    ),
)

path = v3_rollout.write_rollout_artifact(
    OUT_DIR,
    result,
    code_ref=CODE_REF,
    split_csv=SPLIT_CSV,
    checkpoint_dir=CHECKPOINT,
    # 핀을 어떻게 확인했는지 사이드카에 남긴다: 체크아웃이 아니라 파일 digest다.
    extra={
        "code_ref_verified_by": "module_sha256",
        "module_sha256": MODULE_DIGESTS,
    },
)
sidecar = json.loads((path.parent / "rollout.sidecar.json").read_text(encoding="utf-8"))
print(path, result.rollout.shape, result.rollout.dtype)
assert result.rollout.shape == (3851, 5, 5), result.rollout.shape
# head 축이 없다는 것(ADR 0008)은 출력이 아니라 단정으로 확인한다.
assert result.rollout.ndim == 3, result.rollout.shape
print("head axis: none (ADR 0008)")
print(
    "layers rolled:",
    result.attention_layers,
    "| attention heads averaged:",
    result.attention_heads,
)
print("finite:", bool(np.isfinite(result.rollout).all()))
# 행 합은 1 미만이다: <cls>로 간 몫을 rollout 뒤에 떼어내고 재정규화하지 않는다.
print(
    "row mass:",
    float(result.rollout.sum(-1).min()),
    "-",
    float(result.rollout.sum(-1).max()),
)
print("cls mass:", sidecar["cls_mass"])
print("rollout sha256:", sidecar["rollout_sha256"])